# Notebook 04 — Learned Residual Correction

**Repo:** `residual-phase-lock`  
**Notebook:** `04_learned_residual_correction.ipynb`

## Claim

> Residual structure can be learned and used to reduce drift when the residual learner is bounded.

Notebook arc:

```text
01 → residual reveals structure
02 → topology drift appears
03 → known constraint corrects drift
04 → learned residual correction reduces held-out drift
```

This version uses **Fourier features + Ridge regression** instead of a high-degree polynomial residual model. That matters because residual phase-lock should stabilize coherence, not create unbounded extrapolation.

## 1. Setup

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

if os.path.exists("../src"):
    sys.path.append("..")

try:
    from src.export import ExportManager
except ModuleNotFoundError:
    class ExportManager:
        def __init__(self, notebook_id, notebook_slug):
            self.id = notebook_id
            self.slug = notebook_slug
            self.fig_dir = "figures"
            self.results_dir = "results"
            self.docs_dir = "docs"
            os.makedirs(self.fig_dir, exist_ok=True)
            os.makedirs(self.results_dir, exist_ok=True)
            os.makedirs(self.docs_dir, exist_ok=True)

        def save_fig(self, name):
            path = f"{self.fig_dir}/{self.id}_{name}.png"
            plt.savefig(path, dpi=220, bbox_inches="tight")
            print(f"[export:fallback] saved figure: {path}")

        def save_csv(self, df, name):
            path = f"{self.results_dir}/{self.id}_{name}.csv"
            df.to_csv(path, index=False)
            print(f"[export:fallback] saved csv: {path}")

        def save_json(self, obj, name):
            path = f"{self.results_dir}/{self.id}_{name}.json"
            with open(path, "w") as f:
                json.dump(obj, f, indent=2)
            print(f"[export:fallback] saved json: {path}")

        def write_md(self, title, metrics_dict, figure_names, interpretation=None):
            md_path = f"{self.docs_dir}/{self.id}_{self.slug}.md"
            metrics_lines = "\n".join([f"| {k} | {v:.3f} |" for k, v in metrics_dict.items()])
            figure_lines = "\n\n".join([f"![{name}](../figures/{self.id}_{name}.png)" for name in figure_names])
            interpretation_block = ""
            if interpretation:
                interpretation_block = f'''
## Interpretation

```text
{interpretation.strip()}
```
'''
            md = f'''# Notebook {self.id} — {title}

## Results

| Metric | Value |
|--------|------:|
{metrics_lines}

## Figures

{figure_lines}

{interpretation_block}
'''
            with open(md_path, "w") as f:
                f.write(md)
            print(f"[export:fallback] saved markdown: {md_path}")

np.random.seed(45)

NOTEBOOK_ID = "04"
NOTEBOOK_SLUG = "learned_residual_correction"

exp = ExportManager(NOTEBOOK_ID, NOTEBOOK_SLUG)

## 2. Generate signal with hidden residual structure

The baseline model will fit a trend. The residual learner will learn the missing periodic structure from training residuals and apply that correction on held-out data.

In [ ]:
n = 800
x = np.linspace(0, 12, n)

trend = 0.65 * x + 1.25
hidden_structure = (
    0.75 * np.sin(1.7 * x)
    + 0.28 * np.cos(3.4 * x)
    + 0.12 * np.sin(0.65 * x)
)
noise = np.random.normal(0, 0.18, size=n)

y = trend + hidden_structure + noise
X = x.reshape(-1, 1)

# Interleaved train/test split avoids extrapolation traps.
# This tests held-out generalization across the same domain.
train_mask = np.arange(n) % 4 != 0
test_mask = ~train_mask

X_train, X_test = X[train_mask], X[test_mask]
x_train, x_test = x[train_mask], x[test_mask]
y_train, y_test = y[train_mask], y[test_mask]
hidden_train, hidden_test = hidden_structure[train_mask], hidden_structure[test_mask]

data_df = pd.DataFrame({
    "x": x,
    "observed_y": y,
    "trend": trend,
    "hidden_structure": hidden_structure,
    "noise": noise,
    "split": np.where(train_mask, "train", "test"),
})

# Optional raw synthetic data.
exp.save_csv(data_df, "signal_data")

data_df.head()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(x, y, label="observed signal", linewidth=1.2)
plt.plot(x, trend, label="true trend", linewidth=2)
plt.scatter(x_test, y_test, s=8, label="held-out points", alpha=0.6)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Signal with hidden residual structure")
plt.legend()
plt.tight_layout()
exp.save_fig("signal_with_hidden_structure")
plt.show()

## 3. Baseline model: fit only the trend

In [ ]:
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

y_train_hat = baseline_model.predict(X_train)
y_test_hat = baseline_model.predict(X_test)

train_residual = y_train - y_train_hat
test_residual = y_test - y_test_hat

baseline_train_rmse = float(np.sqrt(mean_squared_error(y_train, y_train_hat)))
baseline_test_rmse = float(np.sqrt(mean_squared_error(y_test, y_test_hat)))
baseline_train_r2 = float(r2_score(y_train, y_train_hat))
baseline_test_r2 = float(r2_score(y_test, y_test_hat))

print(f"Baseline train RMSE: {baseline_train_rmse:.4f}")
print(f"Baseline test RMSE:  {baseline_test_rmse:.4f}")
print(f"Baseline train R²:   {baseline_train_r2:.4f}")
print(f"Baseline test R²:    {baseline_test_r2:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(x, y, label="observed signal", linewidth=1.0)
plt.plot(x, baseline_model.predict(X), label="baseline trend fit", linewidth=2)
plt.scatter(x_test, y_test, s=8, label="held-out points", alpha=0.6)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Baseline trend fit leaves structured residual")
plt.legend()
plt.tight_layout()
exp.save_fig("baseline_trend_fit")
plt.show()

## 4. Fourier residual learner

Fourier features are a bounded residual basis for oscillatory structure. Ridge regression controls coefficient growth.

This is the stable learned residual correction:

```text
baseline output + learned residual → corrected output
```

In [ ]:
def fourier_features(x_values, freqs):
    x_values = np.asarray(x_values).reshape(-1)
    features = []
    for f in freqs:
        features.append(np.sin(f * x_values))
        features.append(np.cos(f * x_values))
    return np.vstack(features).T

freqs = np.array([0.65, 1.7, 3.4, 5.1])
Phi_train = fourier_features(x_train, freqs)
Phi_test = fourier_features(x_test, freqs)
Phi_all = fourier_features(x, freqs)

residual_model = Ridge(alpha=1.0)
residual_model.fit(Phi_train, train_residual)

train_residual_hat = residual_model.predict(Phi_train)
test_residual_hat = residual_model.predict(Phi_test)
all_residual_hat = residual_model.predict(Phi_all)

y_train_corrected = y_train_hat + train_residual_hat
y_test_corrected = y_test_hat + test_residual_hat
y_all_corrected = baseline_model.predict(X) + all_residual_hat

corrected_train_rmse = float(np.sqrt(mean_squared_error(y_train, y_train_corrected)))
corrected_test_rmse = float(np.sqrt(mean_squared_error(y_test, y_test_corrected)))
corrected_train_r2 = float(r2_score(y_train, y_train_corrected))
corrected_test_r2 = float(r2_score(y_test, y_test_corrected))

test_rmse_reduction = float(baseline_test_rmse - corrected_test_rmse)
relative_test_rmse_reduction = float(test_rmse_reduction / baseline_test_rmse)

residual_corr_train = float(np.corrcoef(train_residual, train_residual_hat)[0, 1])
residual_corr_test = float(np.corrcoef(test_residual, test_residual_hat)[0, 1])

print(f"Corrected train RMSE: {corrected_train_rmse:.4f}")
print(f"Corrected test RMSE:  {corrected_test_rmse:.4f}")
print(f"Corrected train R²:   {corrected_train_r2:.4f}")
print(f"Corrected test R²:    {corrected_test_r2:.4f}")
print(f"Relative test RMSE reduction: {relative_test_rmse_reduction:.4f}")
print(f"Residual corr train: {residual_corr_train:.4f}")
print(f"Residual corr test:  {residual_corr_test:.4f}")

In [ ]:
residual_learning_df = pd.DataFrame({
    "x": x,
    "split": np.where(train_mask, "train", "test"),
    "true_hidden_structure": hidden_structure,
    "baseline_prediction": baseline_model.predict(X),
    "learned_residual": all_residual_hat,
    "corrected_prediction": y_all_corrected,
})

exp.save_csv(residual_learning_df, "residual_learning_table")

plt.figure(figsize=(10, 5))
plt.plot(x, hidden_structure, label="true hidden structure", linewidth=2)
plt.plot(x, all_residual_hat, label="learned residual correction", linewidth=2)
plt.scatter(x_test, test_residual_hat, s=10, label="held-out learned residual", alpha=0.6)
plt.xlabel("x")
plt.ylabel("residual component")
plt.title("Bounded learned residual correction tracks hidden structure")
plt.legend()
plt.tight_layout()
exp.save_fig("learned_residual_structure")
plt.show()

## 5. Compare baseline and corrected outputs on held-out points

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(x, y, label="observed signal", linewidth=1.0, alpha=0.6)
plt.plot(x, baseline_model.predict(X), label="baseline prediction", linewidth=2)
plt.plot(x, y_all_corrected, label="corrected prediction", linewidth=2)
plt.scatter(x_test, y_test, s=10, label="held-out points", alpha=0.7)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Learned residual correction improves held-out prediction")
plt.legend()
plt.tight_layout()
exp.save_fig("heldout_correction")
plt.show()

In [ ]:
conditions = pd.DataFrame({
    "condition": ["baseline_train", "corrected_train", "baseline_test", "corrected_test"],
    "rmse": [
        baseline_train_rmse,
        corrected_train_rmse,
        baseline_test_rmse,
        corrected_test_rmse,
    ],
    "r2": [
        baseline_train_r2,
        corrected_train_r2,
        baseline_test_r2,
        corrected_test_r2,
    ],
})

exp.save_csv(conditions, "baseline_vs_corrected")

plt.figure(figsize=(8, 4))
labels = ["Train baseline", "Train corrected", "Test baseline", "Test corrected"]
values = [
    baseline_train_rmse,
    corrected_train_rmse,
    baseline_test_rmse,
    corrected_test_rmse,
]
plt.bar(labels, values)
plt.ylabel("RMSE")
plt.title("RMSE before and after bounded learned residual correction")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
exp.save_fig("rmse_before_after")
plt.show()

conditions

## 6. Residuals before and after correction

In [ ]:
corrected_train_residual = y_train - y_train_corrected
corrected_test_residual = y_test - y_test_corrected

residual_compare_df = pd.DataFrame({
    "x": np.concatenate([x_train, x_test]),
    "split": ["train"] * len(x_train) + ["test"] * len(x_test),
    "baseline_residual": np.concatenate([train_residual, test_residual]),
    "corrected_residual": np.concatenate([corrected_train_residual, corrected_test_residual]),
})

exp.save_csv(residual_compare_df, "residuals_before_after")

plt.figure(figsize=(10, 5))
plt.scatter(x_train, train_residual, s=8, label="baseline residual train", alpha=0.35)
plt.scatter(x_test, test_residual, s=12, label="baseline residual test", alpha=0.6)
plt.scatter(x_train, corrected_train_residual, s=8, label="corrected residual train", alpha=0.35)
plt.scatter(x_test, corrected_test_residual, s=12, label="corrected residual test", alpha=0.6)
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("x")
plt.ylabel("residual")
plt.title("Residuals before and after learned correction")
plt.legend()
plt.tight_layout()
exp.save_fig("residuals_before_after")
plt.show()

## 7. Residual spectrum before and after

A good residual correction should reduce structured spectral concentration in the remaining residual.

In [ ]:
def spectral_structure_score(residual, x_axis, top_k=3):
    order = np.argsort(x_axis)
    x_sorted = np.asarray(x_axis)[order]
    r_sorted = np.asarray(residual)[order]
    centered = r_sorted - r_sorted.mean()

    # Approximate uniform spacing for interleaved held-out points.
    dx = np.median(np.diff(x_sorted))
    freqs_out = np.fft.rfftfreq(len(centered), d=dx)
    spectrum = np.abs(np.fft.rfft(centered))
    top_idx = np.argsort(spectrum)[-top_k:][::-1]
    total_energy = np.sum(spectrum**2)
    dominant_energy = np.sum(spectrum[top_idx]**2)
    score = float(dominant_energy / total_energy)
    return freqs_out, spectrum, top_idx, score

freq_base, spec_base, top_base, score_base = spectral_structure_score(test_residual, x_test)
freq_corr, spec_corr, top_corr, score_corr = spectral_structure_score(corrected_test_residual, x_test)

spectrum_compare = pd.DataFrame({
    "frequency": freq_base,
    "baseline_spectrum": spec_base,
    "corrected_spectrum": spec_corr,
})

exp.save_csv(spectrum_compare, "test_residual_spectrum_before_after")

dominant_modes = pd.DataFrame({
    "rank": np.arange(1, 4),
    "baseline_frequency": freq_base[top_base],
    "baseline_amplitude": spec_base[top_base],
    "corrected_frequency": freq_corr[top_corr],
    "corrected_amplitude": spec_corr[top_corr],
})

exp.save_csv(dominant_modes, "dominant_modes_before_after")

structure_score_reduction = float(score_base - score_corr)

print(f"Baseline test residual structure score:  {score_base:.4f}")
print(f"Corrected test residual structure score: {score_corr:.4f}")
print(f"Structure score reduction: {structure_score_reduction:.4f}")

dominant_modes

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(freq_base, spec_base, label="baseline residual spectrum", linewidth=1.5)
plt.plot(freq_corr, spec_corr, label="corrected residual spectrum", linewidth=1.5)
plt.xlim(0, 2.5)
plt.xlabel("frequency")
plt.ylabel("amplitude")
plt.title("Held-out residual spectrum before and after correction")
plt.legend()
plt.tight_layout()
exp.save_fig("residual_spectrum_before_after")
plt.show()

## 8. Guardrail: unstable residual learners can make drift worse

For comparison, we include a high-degree polynomial residual learner. It is not used as the preferred correction. It shows why residual phase-lock needs bounded correction, not arbitrary residual chasing.

In [ ]:
unstable_model = make_pipeline(
    PolynomialFeatures(degree=9, include_bias=False),
    LinearRegression(),
)
unstable_model.fit(X_train, train_residual)

unstable_test_residual_hat = unstable_model.predict(X_test)
y_test_unstable = y_test_hat + unstable_test_residual_hat

unstable_test_rmse = float(np.sqrt(mean_squared_error(y_test, y_test_unstable)))
unstable_test_r2 = float(r2_score(y_test, y_test_unstable))

guardrail = pd.DataFrame({
    "model": ["baseline", "bounded_fourier_ridge", "unstable_polynomial"],
    "test_rmse": [baseline_test_rmse, corrected_test_rmse, unstable_test_rmse],
    "test_r2": [baseline_test_r2, corrected_test_r2, unstable_test_r2],
})

exp.save_csv(guardrail, "guardrail_comparison")

plt.figure(figsize=(8, 4))
plt.bar(guardrail["model"], guardrail["test_rmse"])
plt.ylabel("test RMSE")
plt.title("Guardrail: bounded correction vs unstable residual chasing")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
exp.save_fig("guardrail_comparison")
plt.show()

guardrail

## 9. Summary outputs

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "baseline_train_rmse",
        "corrected_train_rmse",
        "baseline_test_rmse",
        "corrected_test_rmse",
        "test_rmse_reduction",
        "relative_test_rmse_reduction",
        "baseline_test_r2",
        "corrected_test_r2",
        "residual_corr_train",
        "residual_corr_test",
        "baseline_test_residual_structure_score",
        "corrected_test_residual_structure_score",
        "structure_score_reduction",
        "unstable_polynomial_test_rmse",
    ],
    "value": [
        baseline_train_rmse,
        corrected_train_rmse,
        baseline_test_rmse,
        corrected_test_rmse,
        test_rmse_reduction,
        relative_test_rmse_reduction,
        baseline_test_r2,
        corrected_test_r2,
        residual_corr_train,
        residual_corr_test,
        score_base,
        score_corr,
        structure_score_reduction,
        unstable_test_rmse,
    ],
})

exp.save_csv(summary, "summary")
summary_json = {row["metric"]: float(row["value"]) for _, row in summary.iterrows()}
exp.save_json(summary_json, "summary")

summary

## 10. Generate markdown summary

In [ ]:
exp.write_md(
    title="Learned Residual Correction",
    metrics_dict={
        "Baseline test RMSE": baseline_test_rmse,
        "Corrected test RMSE": corrected_test_rmse,
        "Relative test RMSE reduction": relative_test_rmse_reduction,
        "Residual correlation test": residual_corr_test,
        "Baseline residual structure score": score_base,
        "Corrected residual structure score": score_corr,
        "Unstable polynomial test RMSE": unstable_test_rmse,
    },
    figure_names=[
        "signal_with_hidden_structure",
        "learned_residual_structure",
        "heldout_correction",
        "rmse_before_after",
        "residuals_before_after",
        "residual_spectrum_before_after",
        "guardrail_comparison",
    ],
    interpretation="""
residual → learn missing structure
bounded residual learner → correction
coherence stabilizes as structured residual is removed
unbounded residual chasing can destabilize correction
""",
)

## 11. Output export

Run this final cell in Colab to download notebook outputs.

```text
04_learned_residual_correction_outputs.zip
├── figures/
├── results/
└── docs/
```

In [ ]:
ZIP_NAME = "04_learned_residual_correction_outputs.zip"

os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("docs", exist_ok=True)

!zip -r $ZIP_NAME figures results docs

try:
    from google.colab import files
    files.download(ZIP_NAME)
except ImportError:
    print(f"Not running in Colab. Output zip created locally: {ZIP_NAME}")

## 12. Takeaway

```text
residual structure can be learned
bounded learned residuals can reduce drift
phase-lock needs stable correction, not arbitrary residual chasing
```